In [1]:
%%capture
!pip install facenet-pytorch

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

## Libraries

In [3]:
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training

import torch
import torch.nn as nn
from torch.utils.data import Subset

from torchvision import models
from torchvision import transforms
from torchsummary import summary
from PIL import Image

from image_iter import FaceDataset, customSubset
from custom_model import distill_model, distill_model_2
from utils import model_size

import pickle
from tqdm import tqdm
import time

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset

In [4]:
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'

dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)

BATCH_SIZE = 32
trainloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


In [5]:
# ss = customSubset(train_root)
# indices = []
# for i in range(1000, 2000):
#     indices += ss.class_dict[i]
    
# dataset_sub = Subset(dataset, indices)
    
# trainloader = torch.utils.data.DataLoader(dataset_sub, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

## Load Models

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

In [7]:
teacher = InceptionResnetV1(pretrained='casia-webface').to(device)
student = distill_model().to(device)
weight_path = '/home/pj00/projects/Github/small_face_recognition_trcking/model-weights/mobV3_adam_29.pt'
student.load_state_dict(torch.load(weight_path))

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<All keys matched successfully>

## Evaluate the model

In [8]:
teacher.eval()
student.eval()
print('Done')

Done


In [11]:
mse_loss = []
cosine_sim = []
# embeds_teacher = []
# start_teacher = time.time()

for images, _ in tqdm(trainloader):
    with torch.no_grad():
        z1 = teacher(images.to(device, dtype=torch.float32))
        z2 = student(images.to(device, dtype=torch.float32))
    
    with torch.no_grad():
        mse = torch.mean(torch.square(z1-z2), axis=1)
        cs = torch.nn.functional.cosine_similarity(z1, z2)

        mse_loss += list(mse.detach().cpu().numpy())
        cosine_sim += list(cs.detach().cpu().numpy())
#     embeds_teacher.append(z1.detach().cpu())
# end_teacher = time.time()
# print(end_teacher-start_teacher)

100%|█████████████████████████████████████| 15331/15331 [23:53<00:00, 10.69it/s]


In [13]:
print('Mean MSE: {}'.format(np.mean(mse_loss)))
print('Mean Cosine Similarity: {}'.format(np.mean(cosine_sim)))

Mean MSE: 6.634834335272899e-06
Mean Cosine Similarity: 0.99830162525177


In [9]:
mse_loss = []
cosine_sim = []
embeds_student = []
start_student = time.time()

for images, _ in tqdm(trainloader):
#     z1 = teacher(x.to(device, dtype=torch.float32))
    with torch.no_grad():
        z2 = student(images.to(device, dtype=torch.float32))
    
#     with torch.no_grad():
#         mse = torch.mean(torch.square(z1-z2), axis=1)
#         cs = torch.nn.functional.cosine_similarity(z1, z2)

#         mse_loss += list(mse.detach().cpu().numpy())
#         cosine_sim += list(cs.detach().cpu().numpy())
    embeds_student.append(z2.detach().cpu())
    
end_student = time.time()

print(end_student - start_student)

100%|█████████████████████████████████████| 15331/15331 [08:53<00:00, 28.71it/s]

533.944988489151


In [ ]:
cosine_sim = torch.nn.functional.cosine_similarity(torch.cat(embeds_teacher), torch.cat(embeds_student), dim=1)

In [ ]:
mse = torch.nn.functional.mse_loss(torch.cat(embeds_teacher), torch.cat(embeds_student), reduction='none')
mse_per_row = mse.mean(dim=1)
torch.mean(mse_per_row)

In [ ]:

print('Mean MSE: {}'.format(torch.mean(mse_per_row)))
print('Mean Cosine Similarity: {}'.format(torch.mean(cosine_sim)))